# 03 · Uydular Arası Çapraz Kalibrasyon
**Tez:** Güneş Döngüleri 23–25 Boyunca Güneş Patlamaları, Güneş Lekeleri ve Jeomanyetik Etkiler
**Hazırlayan:** Tunahan İnci · **Defter sürümü:** 1.0 (05.11.2026)

**Neden?** 02B, bilimsel kalite verilerde bile uyduların aynı patlamayı farklı ölçtüğünü gösterdi (ör. GOES-15, GOES-16'dan ~%8 düşük). Döngüleri karşılaştırırken her döngüyü farklı uydular temsil ettiği için bu fark sonuca karışabilir. Bu defter tüm uyduları **GOES-16 ölçeğine** getirir.

**Yöntem:**
1. Aynı patlamayı gören her uydu çifti için akı oranının logaritması alınır (≥C1 patlamalar).
2. Model: `ln(ölçülen akı) = ln(gerçek akı) + c_uydu`. Her çiftin medyan log oranı `c_b − c_a` için bir denklem verir. Denklemler ağırlıklı en küçük karelerle (ağırlık √n) çözülür, GOES-16 için c = 0.
3. Güven aralıkları bootstrap ile (200 tekrar) hesaplanır.
4. **Kontroller:** (a) uydu ağı GOES-16'ya bağlı mı, (b) katsayı patlama büyüklüğüne bağlı mı (C / M / X), (c) katsayı yıllar içinde kayıyor mu (dedektör yaşlanması).
5. Düzeltilmiş akılarla döngü tablosu yeniden hesaplanır ve düzeltme öncesiyle karşılaştırılır.

**Bilinen risk:** Eski uydu grubu (GOES 8–12) ile yeni grup (GOES 13–19) arasındaki tek ortak dönem GOES-10 ile GOES-14'ün 2009 sonundaki ~2 aylık çakışması. Bu dönem güneş minimumuna denk geldiği için çok az ortak patlama var. Bölüm 2 bu bağlantının yeterli olup olmadığını açıkça raporlar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import matplotlib.pyplot as plt, matplotlib.dates as mdates
KOK = Path('/content/drive/MyDrive/gunes_tezi')
ISLENMIS, SEKIL = KOK/'data'/'processed', KOK/'figures'
print('Hazır')

In [ ]:
# Saf fonksiyonlar (test edilebilir)
import numpy as np, pandas as pd

def cift_olaylari(tum, esik=1e-6, aki='pik_aki'):
    # Aynı patlamayı (aynı grup) gören her uydu çifti için log akı oranı: ln(F_b / F_a), a < b
    t = tum[['grup', 'uydu', 'pik', aki]].rename(columns={aki: 'f'})
    m = t.merge(t, on='grup', suffixes=('_a', '_b'))
    m = m[(m.uydu_a < m.uydu_b) & (m.f_a >= esik) & (m.f_b >= esik)].copy()
    m['lr'] = np.log(m.f_b / m.f_a)
    m['pik'] = m.pik_a
    return m[['grup', 'pik', 'uydu_a', 'uydu_b', 'f_a', 'f_b', 'lr']].reset_index(drop=True)

def cift_ozeti(m):
    return (m.groupby(['uydu_a', 'uydu_b']).lr
              .agg(medyan='median', n='size',
                   iqr=lambda s: s.quantile(.75) - s.quantile(.25)).reset_index())

def kalibre_coz(ozet, referans=16, min_n=20):
    # Model: ln F_ölçülen(uydu) = ln F_gerçek + c_uydu.  Her çift için c_b - c_a = medyan(lr).
    # Ağırlıklı en küçük kareler, referans uydu c = 0. Referansa bağlanamayan bileşenler ayrıca raporlanır.
    s = ozet[ozet.n >= min_n]
    uydular = sorted(set(s.uydu_a) | set(s.uydu_b) | {referans})
    ebeveyn = {u: u for u in uydular}
    def bul(x):
        while ebeveyn[x] != x:
            ebeveyn[x] = ebeveyn[ebeveyn[x]]; x = ebeveyn[x]
        return x
    for a, b in zip(s.uydu_a, s.uydu_b):
        ebeveyn[bul(a)] = bul(b)
    bilesen = {}
    for u in uydular:
        bilesen.setdefault(bul(u), []).append(u)
    idx = {u: i for i, u in enumerate(uydular)}
    A, y, w = [], [], []
    for r in s.itertuples():
        row = np.zeros(len(uydular)); row[idx[r.uydu_b]] = 1; row[idx[r.uydu_a]] = -1
        A.append(row); y.append(r.medyan); w.append(np.sqrt(r.n))
    bagli = {}
    for kok, uyeler in bilesen.items():
        if referans in uyeler:
            capa = referans
        else:
            say = {u: s[(s.uydu_a == u) | (s.uydu_b == u)].n.sum() for u in uyeler}
            capa = max(say, key=say.get)
        row = np.zeros(len(uydular)); row[idx[capa]] = 1
        A.append(row); y.append(0.0); w.append(1e4)
        for u in uyeler:
            bagli[u] = referans in uyeler
    A, y, w = np.array(A), np.array(y), np.array(w)
    c = np.linalg.lstsq(A * w[:, None], y * w, rcond=None)[0]
    return pd.DataFrame({'uydu': uydular, 'c': c, 'faktor': np.exp(c),
                         'referansa_bagli': [bagli[u] for u in uydular]})

def bootstrap(m, referans=16, min_n=20, tekrar=200, tohum=42):
    rng = np.random.default_rng(tohum)
    gruplar = {k: g.lr.values for k, g in m.groupby(['uydu_a', 'uydu_b'])}
    sonuc = []
    for _ in range(tekrar):
        satir = [(a, b, np.median(rng.choice(v, len(v))), len(v), 0) for (a, b), v in gruplar.items()]
        oz = pd.DataFrame(satir, columns=['uydu_a', 'uydu_b', 'medyan', 'n', 'iqr'])
        sonuc.append(kalibre_coz(oz, referans, min_n).set_index('uydu').faktor)
    b = pd.concat(sonuc, axis=1)
    return pd.DataFrame({'faktor_q025': b.quantile(.025, axis=1), 'faktor_q975': b.quantile(.975, axis=1)})

def kalibrasyonu_uygula(ncei, katsayi):
    f = ncei.uydu.map(katsayi.set_index('uydu').faktor).fillna(1.0)
    out = ncei.copy()
    out['kal_faktor'] = f
    out['pik_aki_kal'] = out.pik_aki / f
    return out

## 1 · Uydu çiftleri
Her çift için ortak patlama sayısı, medyan oran (`exp(medyan)`) ve dağılımın genişliği (IQR).

In [ ]:
tum = pd.read_parquet(ISLENMIS/'flare_ncei_tum_uydular.parquet')
ncei = pd.read_parquet(ISLENMIS/'flare_ncei.parquet')
m = cift_olaylari(tum, esik=1e-6)
oz = cift_ozeti(m)
oz['oran_b/a'] = np.exp(oz.medyan).round(3)
print(f'{len(m):,} çift ölçüm, {len(oz)} uydu çifti (≥C1)')
print(oz[['uydu_a', 'uydu_b', 'n', 'oran_b/a', 'iqr']].round(3).to_string(index=False))

## 2 · Eski ve yeni uydu grubu arasındaki köprü
GOES 8–12 grubunu GOES 13–19 grubuna bağlayan çiftler. ≥C1 eşiğiyle çok az ortak patlama varsa, B sınıfı patlamalar da (≥B1) denenir. B sınıfında arka plan akısı oranı bozabileceği için ayrıca **arka planı çıkarılmış** akıyla (pik − arka plan) karşılaştırma yapılır.

In [ ]:
eski, yeni = {8, 9, 10, 11, 12}, {13, 14, 15, 16, 17, 18, 19}
def kopru(mm):
    return mm[(mm.uydu_a.isin(eski) & mm.uydu_b.isin(yeni)) | (mm.uydu_a.isin(yeni) & mm.uydu_b.isin(eski))]

print('≥C1 köprü çiftleri:')
k1 = kopru(m); print(cift_ozeti(k1).assign(oran=lambda d: np.exp(d.medyan)).round(3).to_string(index=False) if len(k1) else '  yok')

mB = cift_olaylari(tum, esik=1e-7)
kB = kopru(mB)
print('\n≥B1 köprü çiftleri (ham pik akı):')
print(cift_ozeti(kB).assign(oran=lambda d: np.exp(d.medyan)).round(3).to_string(index=False) if len(kB) else '  yok')

tum_net = tum.assign(net=(tum.pik_aki - tum.arka_plan).clip(lower=1e-9))
mBn = cift_olaylari(tum_net, esik=5e-8, aki='net')
kBn = kopru(mBn)
print('\n≥B1 köprü çiftleri (arka planı çıkarılmış akı):')
print(cift_ozeti(kBn).assign(oran=lambda d: np.exp(d.medyan)).round(3).to_string(index=False) if len(kBn) else '  yok')
if len(kBn):
    print('\nKöprüdeki olaylar:')
    print(kBn[['pik', 'uydu_a', 'uydu_b', 'f_a', 'f_b']].assign(oran=np.exp(kBn.lr)).round(3).to_string(index=False))

## 3 · Katsayıların çözümü
**Ana çözüm:** ≥C1 çiftleri, en az 20 ortak patlama. Köprü bu eşiği geçemezse eski grup kendi içinde (GOES-10'a göre) kalibre edilir ve `referansa_bagli = False` olarak işaretlenir.

**Köprülü çözüm:** Bölüm 2'deki ≥B1 net akı köprüsü eklenerek eski grup GOES-16'ya bağlanır. Köprüdeki olay sayısı az olduğu için güven aralığı geniş olacaktır. Hangi çözümün kullanılacağı bu sonuçlara bakılarak karar günlüğüne yazılacak.

`faktor` = uydunun GOES-16'ya göre ölçtüğü değer. 0,93 ise uydu GOES-16'dan %7 düşük ölçüyor, düzeltmede akısı 0,93'e bölünür.

In [ ]:
ana = kalibre_coz(oz, referans=16, min_n=20)
ana = ana.join(bootstrap(m, 16, 20), on='uydu')
print('ANA ÇÖZÜM (≥C1):'); print(ana.round(3).to_string(index=False))

kopru_oz = cift_ozeti(kBn)
birlesik = pd.concat([oz, kopru_oz], ignore_index=True)
min_kopru = int(kopru_oz.n.max()) if len(kopru_oz) else 0
if min_kopru >= 5:
    kpr = kalibre_coz(birlesik, referans=16, min_n=min(20, min_kopru))
    kpr = kpr.join(bootstrap(pd.concat([m, kBn], ignore_index=True), 16, min(20, min_kopru)), on='uydu')
    print(f'\nKÖPRÜLÜ ÇÖZÜM (köprü n = {min_kopru}):'); print(kpr.round(3).to_string(index=False))
else:
    kpr = None
    print(f'\nKöprü için yeterli ortak olay yok (n = {min_kopru}). Eski grup GOES-16 ölçeğine bağlanamıyor.')

ana.to_csv(ISLENMIS/'03_kalibrasyon_katsayilari.csv', index=False)
if kpr is not None: kpr.to_csv(ISLENMIS/'03_kalibrasyon_katsayilari_kopru.csv', index=False)

# Tutarlılık: çözümün her çifti ne kadar iyi açıkladığı (artık)
cz = ana.set_index('uydu').c
oz['tahmin'] = oz.uydu_b.map(cz) - oz.uydu_a.map(cz)
oz['artik_%'] = ((np.exp(oz.medyan - oz.tahmin) - 1) * 100).round(2)
print('\nÇift bazında artık (gözlenen / modellenen − 1, %):')
print(oz[oz.n >= 20][['uydu_a', 'uydu_b', 'n', 'artik_%']].to_string(index=False))

## 4 · Kontroller
**(a) Büyüklüğe bağlılık:** Katsayı C, M ve X sınıfında aynı mı? Farklıysa dedektörler doğrusal değil demektir ve tek katsayı yetmez.
**(b) Zamana bağlılık:** Büyük çiftlerde oran yıllar içinde kayıyor mu?

In [ ]:
m['buyukluk'] = pd.cut(np.sqrt(m.f_a * m.f_b), [1e-6, 1e-5, 1e-4, 1], labels=['C', 'M', 'X'], right=False)
bb = m.groupby(['uydu_a', 'uydu_b', 'buyukluk'], observed=True).lr.agg(['median', 'size']).reset_index()
bb['oran'] = np.exp(bb['median']).round(3)
tablo = bb[bb['size'] >= 10].pivot_table(index=['uydu_a', 'uydu_b'], columns='buyukluk', values='oran', observed=True)
print('(a) Sınıfa göre medyan oran (en az 10 olay):'); print(tablo.to_string())

buyuk = oz[oz.n >= 300][['uydu_a', 'uydu_b']].values.tolist()
fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
for a, b in buyuk:
    s = m[(m.uydu_a == a) & (m.uydu_b == b)]
    y = s.groupby(s.pik.dt.year).lr.agg(['median', 'size'])
    y = y[y['size'] >= 20]
    ax.plot(pd.to_datetime(y.index.astype(str) + '-07-01'), np.exp(y['median']), 'o-', label=f'GOES-{b} / GOES-{a}')
ax.axhline(1, color='grey', ls=':'); ax.set_ylabel('Medyan akı oranı'); ax.grid(alpha=.25)
ax.legend(ncol=3, fontsize=8); ax.set_title('(b) Uydu çiftleri arası oranın yıllara göre değişimi (≥C1, yılda en az 20 olay)')
fig.savefig(SEKIL/'03_oran_zaman.png', dpi=200); plt.show()

## 5 · Düzeltmeyi uygula ve döngüleri yeniden karşılaştır
Üç sürüm yan yana: HEK (operasyonel), NCEI (bilimsel kalite, 02B) ve NCEI + çapraz kalibrasyon. Köprülü çözüm varsa dördüncü sütun olarak o da eklenir. Kalibre edilmiş akıdan sınıflar yeniden hesaplanır; doymuş patlamalar ≥ işaretini korur.

In [ ]:
def harf(aki):
    return pd.cut(np.log10(aki), [-np.inf, -7, -6, -5, -4, np.inf], labels=list('ABCMX'), right=False).astype(str)

kal = kalibrasyonu_uygula(ncei, ana)
kal['sinif_kal'] = harf(kal.pik_aki_kal)
if kpr is not None:
    kk = kalibrasyonu_uygula(ncei, kpr)
    kal['pik_aki_kal_kopru'] = kk.pik_aki_kal
    kal['sinif_kal_kopru'] = harf(kk.pik_aki_kal)
kal.to_parquet(ISLENMIS/'flare_ncei_kalibre.parquet', index=False)

hek = pd.read_parquet(ISLENMIS/'flare_hek_temiz.parquet')
sn = pd.read_parquet(ISLENMIS/'sn_aylik_1749.parquet').set_index('tarih').sn
DONGU = {'23 (2000–2008/11)': ('2000-01-01', '2008-11-30'),
         '24 (2008/12–2019/11)': ('2008-12-01', '2019-11-30'),
         '25 (2019/12–2025)': ('2019-12-01', '2025-12-31')}
def say(df, zaman, sinif, b, s):
    return int(((df[zaman] >= b) & (df[zaman] <= s) & df[sinif].isin(['M', 'X'])).sum())

satir = []
for ad, (b, s) in DONGU.items():
    t_sn = sn[b:s].sum()
    r = {'Döngü': ad, 'Σ SN': round(t_sn),
         'HEK': say(hek, 'event_peaktime', 'sinif', b, s),
         'NCEI': say(kal, 'pik', 'sinif', b, s),
         'NCEI+kal': say(kal, 'pik', 'sinif_kal', b, s)}
    if kpr is not None: r['NCEI+kal(köprü)'] = say(kal, 'pik', 'sinif_kal_kopru', b, s)
    satir.append(r)
dt_ = pd.DataFrame(satir)
for c in [c for c in dt_.columns if c.startswith(('HEK', 'NCEI'))]:
    dt_[c + ' /1000SN'] = (dt_[c] / dt_['Σ SN'] * 1000).round(1)
dt_.to_csv(ISLENMIS/'03_dongu_ozeti_kalibre.csv', index=False)

oranlar = dt_.set_index('Döngü')[[c for c in dt_.columns if c.endswith('/1000SN')]]
print('Döngü 25 / Döngü 23 (leke başına M+X):')
print((oranlar.iloc[2] / oranlar.iloc[0]).round(3).to_string())
print('Döngü 25 / Döngü 24:')
print((oranlar.iloc[2] / oranlar.iloc[1]).round(3).to_string())
dt_